In [ ]:
!pip install stardist
!pip install openslide-python
!pip install openslide-bin

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 4.9 MB/s eta 0:00:00


In [ ]:
from google.colab import files
from google.colab import drive
import os
import sys
from stardist.models import StarDist2D
import openslide
from openslide import OpenSlide
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from stardist.plot import render_label
from csbdeep.utils import normalize
from skimage.measure import regionprops
import json

In [ ]:
print("Montage de Google Drive...")
drive.mount('/content/drive')
file_path = '/content/drive/MyDrive/SLA/Asclepios/svs_slides'
image_name = 'image_test.svs'
image_path = os.path.join(file_path, image_name)

Montage de Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
def slide_processing(slide, top_left_coord, size=(224, 224)):
    """ Load an SVS whole slide image and optional mask.

    Args:
        img_path (str): Path to the SVS image file.
        return_mask (bool): Whether to return an associated mask (if available).

    Returns:
        numpy.ndarray: Image array (and mask if return_mask=True).
    """
    # Extraire la région à partir du niveau de résolution 0 (pleine résolution)
    region_pil = slide.read_region(top_left_coord, level=0, size=size)

    # Convertir l'image PIL en NumPy
    region_np = np.array(region_pil)[:, :, :3]

    return region_np

In [ ]:
dir_path = "/content/drive/MyDrive/SLA/Asclepios/wsi_features"
pos_path = os.path.join(dir_path, '0.position.npy')
feats_path = os.path.join(dir_path, '0.features.0.npy')

# thumbail_mpp = 0.49
mpp_resolution = 0.5

def pos_feat_load(pos_path, feats_path, thumbail_mpp, mpp_resolution=0.5):
    pos = np.load(pos_path)
    feats = np.load(feats_path)
    pos = pos / thumbail_mpp*mpp_resolution

    return pos, feats

In [ ]:
# Extract the dimensions and other properties for each label
def property_extraction(regions):
    nbr_cells = len(regions)
    label_dimensions = []
    for region in regions:
        # You can extract various properties like bounding box, area, etc.
        dimensions = {
            "label": region.label,
            "bounding_box": region.bbox,  # (min_row, min_col, max_row, max_col)
            "area": region.area,
            "centroid": region.centroid,
            "nbr_cells": len(regions)
            # "min_intensity": region.intensity_image.min(),
            # "max_intensity": region.intensity_image.max() if region.intensity_image is not None else None,
        }
        label_dimensions.append(dimensions)

    # Print out the label dimensions
    # print(f"Number of cells {nbr_cells}")
    # for label in label_dimensions:
    #     print(f"Label {label['label']} has centroid {label['centroid']} and area {label['area']}")

    return nbr_cells, label_dimensions

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from stardist.plot import render_label

def display_labels(img, labels):
    plt.figure(figsize=(12, 6))  # Set figure size

    # Display input image
    plt.subplot(1, 2, 1)
    plt.imshow(img, cmap="gray")
    plt.axis("off")
    plt.title("Input Image")

    # Display label overlay
    ax = plt.subplot(1, 2, 2)
    ax.imshow(render_label(labels, img=img))  # Overlay labels on image
    ax.imshow(render_label(labels))
    ax.axis("off")
    ax.set_title("Prediction + Input Overlay")  # Fixed title setting

    # Add bounding boxes and centroids
    for region in regionprops(labels):
        min_row, min_col, max_row, max_col = region.bbox  # Bounding box coordinates

        # Draw bounding box
        rect = patches.Rectangle(
            (min_col, min_row),  # (x, y) in image coordinates
            max_col - min_col,   # Width
            max_row - min_row,   # Height
            linewidth=2, edgecolor='r', facecolor='none'
        )
        ax.add_patch(rect)  # Add bounding box

        # Plot centroid
        ax.plot(region.centroid[1], region.centroid[0], 'ro', markersize=5)  # 'ro' = red circle

    plt.tight_layout()  # Adjust spacing
    plt.show()  # Display the figure


In [ ]:
# creates a pretrained model
model = StarDist2D.from_pretrained('2D_versatile_he')

if os.path.exists(image_path):
    try:
        # Charger l'image .svs avec OpenSlide
        slide = openslide.OpenSlide(image_path)
        print(slide)
        # Lire un aperçu de l'image (taille ajustable)
        thumbnail = slide.get_thumbnail((1024, 1024))  # Aperçu de taille 1024x1024 pixels

        # # Afficher l'image avec matplotlib
        # plt.imshow(thumbnail)
        # plt.axis('off')
        # plt.show()

    except Exception as e:
        print(f"Erreur lors de l'ouverture de l'image .svs avec OpenSlide : {e}")
else:
    print("Le fichier spécifié est introuvable.")

slide = OpenSlide(image_path)
# Extract microns per pixel (mpp) values
mpp_x = float(slide.properties.get("openslide.mpp-x", "0"))  # Default to 0 if not found
mpp_y = float(slide.properties.get("openslide.mpp-y", "0"))
pos, feats = pos_feat_load(pos_path, feats_path, mpp_x)
# print(f"Microns per pixel (X): {mpp_x} µm/px")
# print(f"Microns per pixel (Y): {mpp_y} µm/px")

Found model '2D_versatile_he' for 'StarDist2D'.
5294730/5294730 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Loading network weights from 'weights_best.h5'.
Loading thresholds from 'thresholds.json'.
Using default values: prob_thresh=0.692478, nms_thresh=0.3.
OpenSlide('/content/drive/MyDrive/SLA/Asclepios/svs_slides/image_test.svs')


In [ ]:
print(slide.dimensions)
print(feats.shape)

pos_centers = pos[:, :2] + np.array([112, 112])
pos_centers /= np.array([slide.dimensions[0], slide.dimensions[0]])
pos_centers.astype(int)
print(pos[0])
print(pos_centers[0])

# ajout données spatiale
feats_spatial = np.concatenate((feats, pos), axis=1)
print(feats_spatial.shape)

(47990, 32626)
(6162, 1536)
[32762.97335203   224.40392707 32987.3772791    448.80785414]
[0.68503799 0.00700988]
(6162, 1540)


In [ ]:

metadata_list = []
for i, patch_pos in enumerate(pos):
    patch_x = int(patch_pos[0])
    pos_y = int(patch_pos[1])
    img = slide_processing(slide, (patch_pos_x, patch_pos_y))
    labels, _ = model.predict_instances(normalize(img))
    if i >= 50 and i < 100:
      display_labels(img, labels)
    # Extract label properties using skimage.measure.regionprops
    regions = regionprops(labels)
    nbr_cells, label_dimensions = property_extraction(regions)
    metadata_path = {'patch_num': i+1, 'patch_pos':pos[i].tolist(), 'patch_feats':'cc', 'patch_nbr_cells': nbr_cells, 'cells_data': label_dimensions}
    metadata_list.append(metadata_path)
    if i == 100:
       break

In [ ]:
sprint(metadata_list)

In [ ]:
file_name = "metadata.json"

with open(file_name, 'w') as file:
   json.dump(metadata_list, file, indent=4)

print(f"JSON file '{file_name}' created successfully!")